In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [24]:
df_train = pd.read_csv("C:/Users/mutch_lf652j0/Credit Score Interactive Model/data/processed/cleaned_train.csv")
df_test = pd.read_csv("C:/Users/mutch_lf652j0/Credit Score Interactive Model/data/processed/cleaned_test.csv")

In [25]:
df_train.head()

,age,annual_income,employment_length_years,home_ownership,region,num_open_accounts,num_delinquencies_2yr,total_revolving_balance,credit_utilisation_pct,months_since_oldest_account,...,interest_rate,loan_purpose,dti_ratio,months_since_last_delinquency,pct_accounts_current,application_dow,email_domain_type,default_flag,missing_annual_income,missing_months_since_last_delinquency
0,25.0,32005.0,1.4,MORTGAGE,South-Urban,10.0,0,0.0,31.2,42.0,...,12.69,medical,0.337,105.0,93.0,Friday,free,0,0,0
1,23.9,25010.0,4.0,MORTGAGE,North-Urban,6.0,1,3962.0,70.8,55.0,...,16.56,debt_consolidation,0.294,96.0,84.5,Thursday,corporate,1,0,0
2,52.5,39710.0,7.9,MORTGAGE,East-Urban,6.0,1,1630.0,66.6,281.0,...,13.22,home_improvement,0.214,37.0,57.2,Wednesday,free,0,0,0
3,48.2,49147.0,12.3,RENT,Central-Urban,6.0,0,4414.0,57.3,237.0,...,11.55,medical,0.137,900.0,92.5,Wednesday,corporate,0,0,1
4,45.1,100583.0,3.4,RENT,North-Suburban,8.0,0,2985.0,33.7,224.0,...,8.12,debt_consolidation,0.050,900.0,78.4,Monday,free,0,0,1


In [26]:
df_train.columns.tolist()

['age',
 'annual_income',
 'employment_length_years',
 'home_ownership',
 'region',
 'num_open_accounts',
 'num_delinquencies_2yr',
 'total_revolving_balance',
 'credit_utilisation_pct',
 'months_since_oldest_account',
 'num_hard_inquiries_6mo',
 'loan_amount',
 'interest_rate',
 'loan_purpose',
 'dti_ratio',
 'months_since_last_delinquency',
 'pct_accounts_current',
 'application_dow',
 'email_domain_type',
 'default_flag',
 'missing_annual_income',
 'missing_months_since_last_delinquency']

In [27]:
#lets start with a few helpful financial ratios that we can create from the existing features. Some are commonly used in credit scoring models and should help our model learn some useful patterns in the data.

In [28]:
df_train["months_since_last_delinquency"].nunique()

176

In [29]:

#Monthly debt payments relative to income
df_train['loan_to_income'] = df_train['loan_amount'] / df_train['annual_income']
#Total revolving balance relative to income
df_train['revolving_balance_to_income'] = df_train['total_revolving_balance'] / df_train['annual_income']
#interest cost relative to income
df_train['interest_to_income'] = (df_train['loan_amount'] * df_train['interest_rate']) / df_train['annual_income']

#credit history relative to age, this may solve the multicolinearity issue and create a more useful feature for the model to learn from
df_train["credit_history_to_age"] = df_train["months_since_oldest_account"] / (df_train["age"] *12)

#delinquency rate accross open accounts
df_train["delinquency_rate"] = df_train["num_delinquencies_2yr"] / (df_train["num_open_accounts"] + 1)

#Multiplying effect of hard inquiries and delinquencies.
df_train['hard_inquiries_delinquencies'] = df_train['num_hard_inquiries_6mo'] * df_train['num_delinquencies_2yr']
df_train['hard_inquiries_delinquencies'] = pd.cut(
    df_train['hard_inquiries_delinquencies'],
    bins=[-1, 0, 2, 6, float('inf')],
    labels=[0, 1, 2, 3])

#lets categorise the delinquency risk based on number of months.

delinquent_mask = df_train['months_since_last_delinquency'] != 900

df_train['delinquency_recency_bucket'] = 0  # default = never delinquent

df_train.loc[delinquent_mask, 'delinquency_recency_bucket'] = pd.qcut(
    df_train.loc[delinquent_mask, 'months_since_last_delinquency'],
    q=3,
    labels=[3, 2, 1]  # 3=most recent/risky, 1=oldest/least risky
)

#high dti and high interest is a risky combination.
df_train["rate_dti_burden"] = df_train["interest_rate"] * df_train["dti_ratio"]

#rate multiplied with delinquency risk.  
df_train['rate_delinquency_risk'] = df_train['interest_rate'] * df_train['num_delinquencies_2yr']

#younger borrower with high interest may be riskier.
df_train['rate_to_age'] = df_train['interest_rate'] / df_train['age']







In [30]:
df_train.head(10)

,age,annual_income,employment_length_years,home_ownership,region,num_open_accounts,num_delinquencies_2yr,total_revolving_balance,credit_utilisation_pct,months_since_oldest_account,...,loan_to_income,revolving_balance_to_income,interest_to_income,credit_history_to_age,delinquency_rate,hard_inquiries_delinquencies,delinquency_recency_bucket,rate_dti_burden,rate_delinquency_risk,rate_to_age
0,25.0,32005.0,1.4,MORTGAGE,South-Urban,10.0,0,0.0,31.2,42.0,...,0.259116,0.000000,3.288179,0.140000,0.000000,0,1,4.27653,0.00,0.507600
1,23.9,25010.0,4.0,MORTGAGE,North-Urban,6.0,1,3962.0,70.8,55.0,...,0.882247,0.158417,14.610012,0.191771,0.142857,0,1,4.86864,16.56,0.692887
2,52.5,39710.0,7.9,MORTGAGE,East-Urban,6.0,1,1630.0,66.6,281.0,...,0.229539,0.041048,3.034508,0.446032,0.142857,1,1,2.82908,13.22,0.251810
3,48.2,49147.0,12.3,RENT,Central-Urban,6.0,0,4414.0,57.3,237.0,...,0.132439,0.089812,1.529675,0.409751,0.000000,0,0,1.58235,0.00,0.239627
4,45.1,100583.0,3.4,RENT,North-Suburban,8.0,0,2985.0,33.7,224.0,...,0.114691,0.029677,0.931294,0.413895,0.000000,0,0,0.40600,0.00,0.180044
5,27.9,63077.0,15.9,RENT,East-Urban,9.0,0,3441.0,38.5,67.0,...,0.120678,0.054552,1.345559,0.200119,0.000000,0,0,3.59030,0.00,0.399642
6,41.8,24701.0,11.1,OWN,South-Urban,9.0,1,0.0,21.9,213.0,...,0.551840,0.000000,9.712384,0.424641,0.100000,1,2,4.69920,17.60,0.421053
7,21.0,25493.0,6.4,MORTGAGE,North-Urban,9.0,0,0.0,50.4,79.0,...,0.389833,0.000000,4.385616,0.313492,0.000000,0,0,2.47500,0.00,0.535714
8,48.6,71159.0,15.5,MORTGAGE,North-Urban,13.0,3,8142.0,44.0,232.0,...,0.143060,0.114420,2.410559,0.397805,0.214286,2,2,3.95975,50.55,0.346708
9,21.0,135187.0,7.2,MORTGAGE,East-Suburban,5.0,0,4913.0,38.9,31.0,...,0.126299,0.036342,0.967451,0.123016,0.000000,0,0,0.56684,0.00,0.364762
